In [ ]:
!pip install -q pandas pyarrow

In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
BASE_DIR = Path.cwd()
SHARED_DATA_DIR = (BASE_DIR / "../../data/shared_data").resolve()

STAGING_DIR = SHARED_DATA_DIR / "staging"
MASTER_DIR = SHARED_DATA_DIR / "master"
LOGS_DIR = SHARED_DATA_DIR / "logs"

MASTER_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

VIDEOS_MASTER_FILE = MASTER_DIR / "videos_master.parquet"
COMMENTS_MASTER_FILE = MASTER_DIR / "comments_master.parquet"
RUNS_MASTER_FILE = LOGS_DIR / "runs_log_master.parquet"

In [ ]:
def load_parquet_if_exists(path: Path) -> pd.DataFrame:
    if path.exists() and path.stat().st_size > 0:
        return pd.read_parquet(path)
    return pd.DataFrame()

def deduplicate_videos(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    x = df.copy()
    x["video_published_at"] = pd.to_datetime(x["video_published_at"], utc=True, errors="coerce")
    x["fetched_at_utc"] = pd.to_datetime(x["fetched_at_utc"], utc=True, errors="coerce")
    x = x.sort_values(["video_published_at", "fetched_at_utc"], ascending=[False, False], na_position="last")
    x = x.drop_duplicates(subset=["video_id"], keep="first")
    return x.reset_index(drop=True)

def deduplicate_comments(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    x = df.copy()
    x["published_at"] = pd.to_datetime(x["published_at"], utc=True, errors="coerce")
    x["updated_at"] = pd.to_datetime(x["updated_at"], utc=True, errors="coerce")
    x["fetched_at_utc"] = pd.to_datetime(x["fetched_at_utc"], utc=True, errors="coerce")
    x = x.sort_values(["updated_at", "published_at", "fetched_at_utc"], ascending=[False, False, False], na_position="last")
    x = x.drop_duplicates(subset=["comment_id"], keep="first")
    return x.reset_index(drop=True)

def merge_master_with_new(master: pd.DataFrame, new_df: pd.DataFrame, kind: str) -> pd.DataFrame:
    if master.empty and new_df.empty:
        return pd.DataFrame()

    if master.empty:
        combined = new_df.copy()
    elif new_df.empty:
        combined = master.copy()
    else:
        combined = pd.concat([master, new_df], ignore_index=True)

    if kind == "videos":
        return deduplicate_videos(combined)
    elif kind == "comments":
        return deduplicate_comments(combined)
    else:
        return combined.drop_duplicates().reset_index(drop=True)

In [ ]:
video_files = sorted(STAGING_DIR.glob("*_videos_*.parquet"))
comment_files = sorted(STAGING_DIR.glob("*_comments_*.parquet"))
run_files = sorted(STAGING_DIR.glob("*_runs_*.parquet"))

print("Video staging files:", len(video_files))
print("Comment staging files:", len(comment_files))
print("Run log staging files:", len(run_files))

for f in video_files[:5]:
    print("VIDEO:", f.name)

for f in comment_files[:5]:
    print("COMMENT:", f.name)

for f in run_files[:5]:
    print("RUN:", f.name)

In [ ]:
video_dfs = [pd.read_parquet(f) for f in video_files] if video_files else []
comment_dfs = [pd.read_parquet(f) for f in comment_files] if comment_files else []
run_dfs = [pd.read_parquet(f) for f in run_files] if run_files else []

staging_videos = pd.concat(video_dfs, ignore_index=True) if video_dfs else pd.DataFrame()
staging_comments = pd.concat(comment_dfs, ignore_index=True) if comment_dfs else pd.DataFrame()
staging_runs = pd.concat(run_dfs, ignore_index=True) if run_dfs else pd.DataFrame()

print("Staging videos rows:", 0 if staging_videos.empty else len(staging_videos))
print("Staging comments rows:", 0 if staging_comments.empty else len(staging_comments))
print("Staging runs rows:", 0 if staging_runs.empty else len(staging_runs))

In [ ]:
videos_master_old = load_parquet_if_exists(VIDEOS_MASTER_FILE)
comments_master_old = load_parquet_if_exists(COMMENTS_MASTER_FILE)
runs_master_old = load_parquet_if_exists(RUNS_MASTER_FILE)

print("Existing master videos:", 0 if videos_master_old.empty else len(videos_master_old))
print("Existing master comments:", 0 if comments_master_old.empty else len(comments_master_old))
print("Existing master runs:", 0 if runs_master_old.empty else len(runs_master_old))

In [ ]:
videos_master_new = merge_master_with_new(videos_master_old, staging_videos, kind="videos")
comments_master_new = merge_master_with_new(comments_master_old, staging_comments, kind="comments")

if runs_master_old.empty and staging_runs.empty:
    runs_master_new = pd.DataFrame()
elif runs_master_old.empty:
    runs_master_new = staging_runs.drop_duplicates().reset_index(drop=True)
elif staging_runs.empty:
    runs_master_new = runs_master_old.drop_duplicates().reset_index(drop=True)
else:
    runs_master_new = pd.concat([runs_master_old, staging_runs], ignore_index=True).drop_duplicates().reset_index(drop=True)

print("Merged master videos:", 0 if videos_master_new.empty else len(videos_master_new))
print("Merged master comments:", 0 if comments_master_new.empty else len(comments_master_new))
print("Merged master runs:", 0 if runs_master_new.empty else len(runs_master_new))

In [ ]:
if not videos_master_new.empty:
    print("Unique video_id:", videos_master_new["video_id"].nunique(), " / rows:", len(videos_master_new))

if not comments_master_new.empty:
    print("Unique comment_id:", comments_master_new["comment_id"].nunique(), " / rows:", len(comments_master_new))

In [ ]:
display(videos_master_new.head())
display(comments_master_new.head())
display(runs_master_new.head())